# TFT v2 Training — WTI News-Driven Liquidity

Ablation study for thesis §4.3.7/§4.3.8. Three variants, one notebook — toggle `ABLATION_VARIANT` in Cell 3.

| Variant | Target(s) | Max horizon | Categoricals | Entity flags |
|---------|-----------|-------------|--------------|:------------:|
| `v2.0` | `log_volume` | 1 h | int-encoded reals | ✗ |
| `v2.1` | `log_volume` | 1 h | proper categoricals | ✗ |
| `v2.2` | `log_volume`, `amihud`, `price_range` | 28 h | proper categoricals | ✓ (71) |

## Run sequence
1. Set `ABLATION_VARIANT = 'v2.0'` → run all cells end-to-end.
2. Change to `'v2.1'` → re-run from **Cell 3** (no kernel restart needed).
3. Same for `'v2.2'`.

Artifacts land in `04_outputs/tft_v2/{variant}/` after each run.

## Cell 2 — Imports and project root

In [1]:
import os, sys, json, re, pickle, sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss, MultiLoss
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
import warnings
warnings.filterwarnings('ignore')

# --- Path detection (Colab vs local) ---
_on_colab = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
if _on_colab:
    # Adjust if your Drive folder differs
    PROJECT_ROOT = Path('/content/drive/MyDrive/thesis')
    DB_PATH      = PROJECT_ROOT / 'wti_thesis.db'
    MODELS_DIR   = PROJECT_ROOT / 'models'
    OUTPUTS_DIR  = PROJECT_ROOT / 'tft_v2_outputs'
else:
    PROJECT_ROOT = Path('..').resolve()
    DB_PATH      = PROJECT_ROOT / '01_data' / 'wti_thesis.db'
    MODELS_DIR   = PROJECT_ROOT / '01_data' / 'models'
    OUTPUTS_DIR  = PROJECT_ROOT / '04_outputs' / 'tft_v2'

sys.path.insert(0, str(PROJECT_ROOT / '03_src'))
from tft.config import (
    TOTAL_HOURS, ENCODER_LENGTH, MAX_PREDICTION_LENGTH,
    TRAIN_END, VAL_START, VAL_END, TEST_START, TEST_END,
    WAR_ONSET_IDX, WAR_ONSET_DATETIME,
    verify_against_db,
    entity_to_column_name, ENTITY_COL_MAP, COL_TO_ENTITY,
)

pl.seed_everything(42, workers=True)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DB_PATH      : {DB_PATH}  (exists={DB_PATH.exists()})')
print(f'PyTorch      : {torch.__version__}')
print(f'GPU          : {torch.cuda.is_available()} | MPS: {torch.backends.mps.is_available()}')
print(f'Canonical entities: {len(ENTITY_COL_MAP)}')

Seed set to 42


PROJECT_ROOT : /Users/efavila/Documents/Code/hammer-market-intelligence-internship
DB_PATH      : /Users/efavila/Documents/Code/hammer-market-intelligence-internship/01_data/wti_thesis.db  (exists=True)
PyTorch      : 2.11.0
GPU          : False | MPS: True
Canonical entities: 71


## Cell 3 — Ablation variant toggle and config

In [2]:
# ── TOGGLE THIS ──────────────────────────────────────────────────────────────
ABLATION_VARIANT = 'v2.0'   # 'v2.0' | 'v2.1' | 'v2.2'
# ─────────────────────────────────────────────────────────────────────────────

assert ABLATION_VARIANT in ('v2.0', 'v2.1', 'v2.2'), f'Unknown variant: {ABLATION_VARIANT}'

verify_against_db(str(DB_PATH))

VARIANT_CONFIGS = {
    'v2.0': {
        'targets':               ['log_volume'],
        'max_prediction_length': 1,
        'proper_categoricals':   False,
        'entity_flags':          False,
    },
    'v2.1': {
        'targets':               ['log_volume'],
        'max_prediction_length': 1,
        'proper_categoricals':   True,
        'entity_flags':          False,
    },
    'v2.2': {
        'targets':               ['log_volume', 'amihud', 'price_range'],
        'max_prediction_length': MAX_PREDICTION_LENGTH,  # 28
        'proper_categoricals':   True,
        'entity_flags':          True,
    },
}
cfg = VARIANT_CONFIGS[ABLATION_VARIANT]

# Evaluation horizons for v2.2 multi-horizon metrics (0-indexed)
EVAL_HORIZON_INDICES = [0, 2, 5, 11, 27]   # → 1h, 3h, 6h, 12h, 28h
EVAL_HORIZON_LABELS  = [1, 3, 6, 12, 28]

VARIANT_OUT_DIR = OUTPUTS_DIR / ABLATION_VARIANT
(VARIANT_OUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Variant              : {ABLATION_VARIANT}')
print(f'  targets            : {cfg["targets"]}')
print(f'  max_pred_length    : {cfg["max_prediction_length"]}')
print(f'  proper_categoricals: {cfg["proper_categoricals"]}')
print(f'  entity_flags       : {cfg["entity_flags"]}')
print(f'Output dir           : {VARIANT_OUT_DIR}')

Variant              : v2.0
  targets            : ['log_volume']
  max_pred_length    : 1
  proper_categoricals: False
  entity_flags       : False
Output dir           : /Users/efavila/Documents/Code/hammer-market-intelligence-internship/04_outputs/tft_v2/v2.0


## Cell 4 — Load and aggregate modeling data

Builds one row per trading hour:
- Continuous LLM features: mean across usable articles in that hour
- Dominant categoricals: event_type[0] and time_horizon from the highest-magnitude article
- Entity flags (v2.2): binary MAX across articles
- No-news hours: continuous → 0.0, categoricals → `'no_news'`, flags → 0

In [17]:
# entity_to_column_name, ENTITY_COL_MAP, COL_TO_ENTITY imported from tft.config
ENT_COLUMNS = list(ENTITY_COL_MAP.values())
ENT_MAP     = ENTITY_COL_MAP

conn = sqlite3.connect(DB_PATH)

# Market context backbone (all 11,232 trading hours)
df_mc = pd.read_sql(
    'SELECT datetime_hour, log_volume, price_range, log_return, amihud, dxy, vix '
    'FROM market_context ORDER BY datetime_hour',
    conn
)
df_mc['datetime_hour'] = pd.to_datetime(df_mc['datetime_hour'], utc=True)
assert len(df_mc) == TOTAL_HOURS, f'Expected {TOTAL_HOURS} hours, got {len(df_mc)}'

# Usable article LLM features (one row per article)
df_arts = pd.read_sql("""
    SELECT l.article_id, l.datetime_hour,
           l.sentiment_score, l.magnitude, l.certainty,
           l.supply_impact, l.demand_impact, l.risk_premium,
           l.event_type, l.time_horizon
    FROM liquidity l
    WHERE l.usable = 1
""", conn)
df_arts['datetime_hour'] = pd.to_datetime(df_arts['datetime_hour'], utc=True)

# Entity mention rows (only for usable articles via article_entities)
df_ent = pd.read_sql(
    'SELECT article_id, canonical_entity FROM article_entities',
    conn
)

conn.close()

# Parse event_type JSON array → primary category
def parse_event_type(val):
    if not val:
        return 'other'
    try:
        arr = json.loads(val)
        return arr[0] if arr else 'other'
    except (json.JSONDecodeError, TypeError):
        return str(val)

df_arts['event_type_primary'] = df_arts['event_type'].apply(parse_event_type)

# --- Hourly aggregation ---

# Dominant article per hour = highest magnitude, with article_id as deterministic tie-breaker
df_arts_sorted = df_arts.sort_values(
    ['datetime_hour', 'magnitude', 'article_id'],
    ascending=[True, False, True]  # earliest hour, highest magnitude, lowest article_id wins on tie
)
df_dom = df_arts_sorted.groupby('datetime_hour').first()[['event_type_primary', 'time_horizon']]

df_hourly_llm = (
    df_arts
    .groupby('datetime_hour')
    .agg(
        n_articles    = ('article_id',    'count'),
        sentiment_score = ('sentiment_score', 'mean'),
        supply_impact = ('supply_impact',  'mean'),
        demand_impact = ('demand_impact',  'mean'),
        risk_premium  = ('risk_premium',   'mean'),
        magnitude     = ('magnitude',      'mean'),
        certainty     = ('certainty',      'mean'),
    )
    .join(df_dom)
    .reset_index()
)

# --- Entity flag matrix: datetime_hour × entity_col ---
df_ent['ent_col'] = df_ent['canonical_entity'].map(ENT_MAP)
df_ent_valid = df_ent.dropna(subset=['ent_col'])

df_ent_valid = df_ent_valid.merge(
    df_arts[['article_id', 'datetime_hour']].drop_duplicates('article_id'),
    on='article_id', how='left'
)
df_ent_valid = df_ent_valid.dropna(subset=['datetime_hour'])

df_ent_flags = (
    df_ent_valid
    .groupby(['datetime_hour', 'ent_col'])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)
    .reset_index()
)
df_ent_flags.columns.name = None
for col in ENT_COLUMNS:
    if col not in df_ent_flags.columns:
        df_ent_flags[col] = 0
df_ent_flags = df_ent_flags[['datetime_hour'] + ENT_COLUMNS]

# --- Build final per-hour DataFrame ---
df = df_mc.copy()
df = df.merge(df_hourly_llm, on='datetime_hour', how='left')
df = df.merge(df_ent_flags,  on='datetime_hour', how='left')

# Fill no-news hours
CONT_LLM = ['sentiment_score', 'supply_impact', 'demand_impact',
             'risk_premium', 'magnitude', 'certainty']
df[CONT_LLM]  = df[CONT_LLM].fillna(0.0)
df['n_articles']         = df['n_articles'].fillna(0).astype(float)
df['event_type_primary'] = df['event_type_primary'].fillna('no_news')
df['time_horizon']       = df['time_horizon'].fillna('no_news')
for col in ENT_COLUMNS:
    df[col] = df[col].fillna(0).astype(float)

# Calendar features
df['hour']         = df['datetime_hour'].dt.hour.astype(float)
df['day_of_week']  = df['datetime_hour'].dt.dayofweek.astype(float)
df['month']        = df['datetime_hour'].dt.month.astype(float)
df['is_us_session']= ((df['datetime_hour'].dt.hour >= 13) &
                      (df['datetime_hour'].dt.hour <= 21)).astype(float)
df['is_wednesday'] = (df['datetime_hour'].dt.dayofweek == 2).astype(float)

# pytorch-forecasting required columns
df = df.sort_values('datetime_hour').reset_index(drop=True)
df['time_idx'] = df.index
df['asset']    = 'WTI'

# Int-encoded fallback columns for v2.0
EVENT_TYPE_INT = {'no_news':0,'geopolitical':1,'supply':2,'demand':3,'macro':4,'technical':5,'other':6}
TIME_HORIZON_INT = {'no_news':0,'immediate':1,'short_term':2,'structural':3,'long_term':3}
df['event_type_int']   = df['event_type_primary'].map(EVENT_TYPE_INT).fillna(6).astype(float)
df['time_horizon_int'] = df['time_horizon'].map(TIME_HORIZON_INT).fillna(0).astype(float)


# --- Verify dominant-article selection (5 sample hours with >1 article) ---
_multi = df_arts.groupby('datetime_hour').filter(lambda g: len(g) > 1)
if len(_multi) > 0:
    _sample_hours = _multi['datetime_hour'].drop_duplicates().iloc[:5]
    print('\nDominant-article verification (hour → chosen article vs candidates):')
    for _h in _sample_hours:
        _grp = df_arts[df_arts['datetime_hour'] == _h][[
             'magnitude', 'article_id', 'event_type_primary', 'time_horizon'
        ]].sort_values('magnitude', ascending=False)
        _chosen = _grp.iloc[0]
        print(f'  {_h}')
        print(f'    CHOSEN  id={int(_chosen.article_id):>6}  mag={_chosen.magnitude:.2f}  '
              f'et={_chosen.event_type_primary}  th={_chosen.time_horizon}')
        for _, _r in _grp.iloc[1:].iterrows():
            print(f'    other   id={int(_r.article_id):>6}  mag={_r.magnitude:.2f}  '
                  f'et={_r.event_type_primary}  th={_r.time_horizon}')
else:
    print('No multi-article hours found in sample — skipping verification')

# Forward-fill sparse macro covariates (≤10 nulls in DXY/VIX due to yfinance gaps
# on holidays/half-sessions where WTI futures traded but the underlying index didn't update)
n_dxy_null = df['dxy'].isna().sum()
n_vix_null = df['vix'].isna().sum()
df['dxy'] = df['dxy'].ffill().bfill()  # ffill primary; bfill handles any null at row 0
df['vix'] = df['vix'].ffill().bfill()

# Returns can't be computed on the first hour (no prior price); fill the boundary NaN with 0
n_logret_null = df['log_return'].isna().sum()
df['log_return'] = df['log_return'].fillna(0.0)
df['amihud'] = df['amihud'].fillna(0.0)

print(f'Macro covariates forward-filled: {n_dxy_null} DXY, {n_vix_null} VIX nulls patched')


# Sanity
assert len(df) == TOTAL_HOURS
assert df[['log_volume','dxy','vix','log_return','amihud']].isna().sum().sum() == 0, (
    f'Unexpected nulls after fill: '
    f'log_volume={df["log_volume"].isna().sum()}, '
    f'dxy={df["dxy"].isna().sum()}, '
    f'vix={df["vix"].isna().sum()}, '
    f'log_return={df["log_return"].isna().sum()}'
)

print(f'Hourly DataFrame : {len(df):,} rows × {len(df.columns)} columns')
print(f'Hours with news  : {(df["n_articles"] > 0).sum():,} / {len(df):,}')
print(f'event_type_primary distribution:')
print(df['event_type_primary'].value_counts().to_string())


Dominant-article verification (hour → chosen article vs candidates):
  2024-05-13 12:00:00+00:00
    CHOSEN  id=  3005  mag=0.90  et=geopolitical  th=immediate
    other   id=   154  mag=0.90  et=geopolitical  th=structural
    other   id=  2724  mag=0.85  et=supply  th=short_term
    other   id=  2932  mag=0.85  et=geopolitical  th=immediate
    other   id=  3007  mag=0.85  et=geopolitical  th=immediate
    other   id=  2469  mag=0.85  et=supply  th=short_term
    other   id=  3053  mag=0.85  et=geopolitical  th=short_term
    other   id=   267  mag=0.85  et=geopolitical  th=structural
    other   id=  2663  mag=0.80  et=geopolitical  th=immediate
    other   id=  2069  mag=0.80  et=geopolitical  th=short_term
    other   id=  2146  mag=0.80  et=geopolitical  th=short_term
    other   id=  2911  mag=0.80  et=geopolitical  th=immediate
    other   id=    71  mag=0.80  et=geopolitical  th=immediate
    other   id=  1022  mag=0.80  et=supply  th=structural
    other   id=  2712  mag=0.8

## Cell 5 — Train / val / test split

In [19]:
# Splits are locked in config — do not change
print(f'Total hours  : {TOTAL_HOURS:,}')
print(f'Train        : time_idx   0 – {TRAIN_END-1}  ({TRAIN_END:,} rows)')
print(f'[buffer]     : time_idx {TRAIN_END} – {VAL_START-1}  ({VAL_START-TRAIN_END} rows, encoder context)')
print(f'Val          : time_idx {VAL_START} – {VAL_END-1}  ({VAL_END-VAL_START:,} rows)')
print(f'[buffer]     : time_idx {VAL_END} – {TEST_START-1}  ({TEST_START-VAL_END} rows, encoder context)')
print(f'Test         : time_idx {TEST_START} – {TEST_END-1}  ({TEST_END-TEST_START:,} rows)')
print(f'  Pre-war    : time_idx {TEST_START} – {WAR_ONSET_IDX-1}  ({WAR_ONSET_IDX-TEST_START:,} rows)')
print(f'  War        : time_idx {WAR_ONSET_IDX} – {TEST_END-1}  ({TEST_END-WAR_ONSET_IDX:,} rows)')
print(f'  War onset  : {WAR_ONSET_DATETIME}')

Total hours  : 11,232
Train        : time_idx   0 – 7861  (7,862 rows)
[buffer]     : time_idx 7862 – 7909  (48 rows, encoder context)
Val          : time_idx 7910 – 9546  (1,637 rows)
[buffer]     : time_idx 9547 – 9594  (48 rows, encoder context)
Test         : time_idx 9595 – 11231  (1,637 rows)
  Pre-war    : time_idx 9595 – 10055  (461 rows)
  War        : time_idx 10056 – 11231  (1,176 rows)
  War onset  : 2026-03-01 23:00:00+00:00


## Cell 6 — Construct TimeSeriesDataSet objects

In [22]:
# Shared kwargs for all variants
SHARED_KW = dict(
    time_idx             = 'time_idx',
    group_ids            = ['asset'],
    max_encoder_length   = ENCODER_LENGTH,
    allow_missing_timesteps = False,
    add_relative_time_idx   = True,
    add_target_scales       = True,
    add_encoder_length      = True,
    static_categoricals     = ['asset'],
    time_varying_known_reals = [
        'hour', 'day_of_week', 'month', 'is_us_session', 'is_wednesday'
    ],
)

# Variant-specific configuration
BASE_UNKNOWN_REALS = [
    'log_volume', 'price_range', 'log_return', 'amihud', 'dxy', 'vix',
    'sentiment_score', 'supply_impact', 'demand_impact', 'risk_premium',
    'magnitude', 'certainty', 'n_articles',
]

if ABLATION_VARIANT == 'v2.0':
    VARIANT_KW = dict(
        target                          = 'log_volume',
        max_prediction_length           = 1,
        time_varying_unknown_reals      = BASE_UNKNOWN_REALS + ['event_type_int', 'time_horizon_int'],
        time_varying_unknown_categoricals = [],
        time_varying_known_categoricals = [],
    )
elif ABLATION_VARIANT == 'v2.1':
    VARIANT_KW = dict(
        target                          = 'log_volume',
        max_prediction_length           = 1,
        time_varying_unknown_reals      = BASE_UNKNOWN_REALS,
        # News categoricals are unknown: future hour's news is not available at
        # prediction time; decoder sees only encoder history + calendar features.
        time_varying_unknown_categoricals = ['event_type_primary', 'time_horizon'],
        time_varying_known_categoricals = [],
    )
else:  # v2.2
    VARIANT_KW = dict(
        target                          = ['log_volume', 'amihud', 'price_range'],
        max_prediction_length           = MAX_PREDICTION_LENGTH,
        time_varying_unknown_reals      = BASE_UNKNOWN_REALS + ENT_COLUMNS,
        # Same rationale: future news features unknown during the 28h decoder window.
        time_varying_unknown_categoricals = ['event_type_primary', 'time_horizon'],
        time_varying_known_categoricals = [],
    )

# Training dataset (rows 0 to TRAIN_END inclusive)
train_data = df[df['time_idx'] <= TRAIN_END].copy()
train_dataset = TimeSeriesDataSet(train_data, **SHARED_KW, **VARIANT_KW)

def make_eval_dataset(training_ds, df_full, min_pred_idx, max_pred_idx):
    """Create an eval dataset with encoder context starting before min_pred_idx
    and prediction targets ending at max_pred_idx."""
    context_start = max(0, min_pred_idx - ENCODER_LENGTH)
    return TimeSeriesDataSet.from_dataset(
        training_ds,
        df_full[(df_full['time_idx'] >= context_start) &
                (df_full['time_idx'] < max_pred_idx)].copy(),
        min_prediction_idx = min_pred_idx,
        predict            = False,
        stop_randomization = True,
    )

val_dataset  = make_eval_dataset(train_dataset, df, VAL_START, VAL_END)
test_dataset = make_eval_dataset(train_dataset, df, TEST_START, TEST_END)

print(f'Train samples : {len(train_dataset):,}')
print(f'Val samples   : {len(val_dataset):,}')
print(f'Test samples  : {len(test_dataset):,}')

Train samples : 7,815
Val samples   : 1,637
Test samples  : 1,637


## Cell 7 — Instantiate TemporalFusionTransformer

In [23]:
if isinstance(VARIANT_KW['target'], list):
    loss = MultiLoss([QuantileLoss()] * len(VARIANT_KW['target']))
else:
    loss = QuantileLoss()

tft = TemporalFusionTransformer.from_dataset(
    train_dataset,
    hidden_size               = 32,
    attention_head_size       = 4,
    dropout                   = 0.1,
    hidden_continuous_size    = 16,
    learning_rate             = 1e-3,
    loss                      = loss,
    reduce_on_plateau_patience= 3,
    log_interval              = 10,
    log_val_interval          = 1,
)

n_params = sum(p.numel() for p in tft.parameters() if p.requires_grad)
print(f'TFT {ABLATION_VARIANT}: {n_params:,} trainable parameters')
print(f'Loss: {loss.__class__.__name__}')

TFT v2.0: 118,114 trainable parameters
Loss: QuantileLoss


## Cell 8 — Trainer setup

In [26]:
BATCH_SIZE = 128 if (torch.cuda.is_available() or torch.backends.mps.is_available()) else 64

train_loader = train_dataset.to_dataloader(train=True,  batch_size=BATCH_SIZE, num_workers=0)
val_loader   = val_dataset.to_dataloader(  train=False, batch_size=BATCH_SIZE, num_workers=0)

callbacks = [
    EarlyStopping(
        monitor   = 'val_loss',
        patience  = 5,
        min_delta = 1e-4,
        mode      = 'min',
    ),
    ModelCheckpoint(
        monitor   = 'val_loss',
        save_top_k= 1,
        mode      = 'min',
        dirpath   = str(MODELS_DIR),
        filename  = f'tft_{ABLATION_VARIANT}-{{epoch}}-{{val_loss:.4f}}',
    ),
]

if torch.cuda.is_available():
    accelerator = 'gpu'
elif torch.backends.mps.is_available():
    accelerator = 'mps'
else:
    accelerator = 'cpu'

trainer = pl.Trainer(
    max_epochs       = 50,
    gradient_clip_val= 0.1,
    accelerator      = accelerator,
    devices          = 1,
    callbacks        = callbacks,
    logger           = CSVLogger(save_dir=str(VARIANT_OUT_DIR / 'logs'), name='', version=''),
    deterministic    = True,
    enable_progress_bar = True,
)

print(f'Accelerator  : {accelerator}')
print(f'Batch size   : {BATCH_SIZE}')
print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Accelerator  : mps
Batch size   : 128
Train batches: 61
Val batches  : 13


## Cell 9 — Train

Expected duration on Colab Pro A100: ~30 min for v2.0/v2.1, ~1.5–2 h for v2.2.

In [ ]:
trainer.fit(tft, train_loader, val_loader)

## Cell 10 — Save best checkpoint path

In [ ]:
best_path      = trainer.checkpoint_callback.best_model_path
best_val_loss  = float(trainer.checkpoint_callback.best_model_score)
best_epoch     = trainer.current_epoch

print(f'Best checkpoint : {best_path}')
print(f'Best val_loss   : {best_val_loss:.4f}')
print(f'Stopped at epoch: {best_epoch}')

## Cell 11 — Load best checkpoint for evaluation

In [ ]:
tft_eval = TemporalFusionTransformer.load_from_checkpoint(best_path)
tft_eval.eval()
print(f'Loaded: {Path(best_path).name}')

## Cell 12 — Generate predictions on val and test sets

In [ ]:
test_loader = test_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)

with torch.no_grad():
    val_raw  = tft_eval.predict(val_loader,  mode='raw', return_x=True,
                                return_index=True, return_y=True)
    test_raw = tft_eval.predict(test_loader, mode='raw', return_x=True,
                                return_index=True, return_y=True)

_pred = val_raw.output['prediction']
_pred_repr = {k: v.shape for k, v in _pred.items()} if isinstance(_pred, dict) else _pred.shape
print(f'Val prediction  : {_pred_repr}')
print(f'Output keys     : {list(val_raw.output.keys())}')

# Recover test time_idx for pre-war / war split
if hasattr(test_raw, 'index') and test_raw.index is not None:
    _idx = test_raw.index
    if isinstance(_idx, pd.DataFrame):
        test_time_idx = _idx['time_idx'].values
    elif isinstance(_idx, dict):
        test_time_idx = np.asarray(_idx['time_idx'])
    else:
        test_time_idx = None
else:
    test_time_idx = None

if test_time_idx is not None:
    prewar_mask = test_time_idx <  WAR_ONSET_IDX
    war_mask    = test_time_idx >= WAR_ONSET_IDX
    print(f'Test pre-war samples: {prewar_mask.sum():,}')
    print(f'Test war samples    : {war_mask.sum():,}')
else:
    # Approximate split by proportion
    n_test = len(list(_pred.values())[0]) if isinstance(_pred, dict) else len(_pred)
    _split = int((WAR_ONSET_IDX - TEST_START) / (TEST_END - TEST_START) * n_test)
    prewar_mask = np.zeros(n_test, dtype=bool); prewar_mask[:_split] = True
    war_mask    = ~prewar_mask
    print(f'Warning: time_idx not available, using approximate split at sample {_split}')

## Cell 13 — Compute metrics

`(target, horizon, slice)` → MAE, RMSE. Saved to `metrics.csv`.

In [ ]:
def _get_pred_actual(raw_out, target_key, horizon_idx):
    """Return (pred_median, actuals) numpy arrays for one target and horizon."""
    pred = raw_out.output['prediction']
    y    = raw_out.y[0]

    if isinstance(pred, (list, tuple)):
        # Multi-target: list order matches target list in dataset
        t_idx = cfg['targets'].index(target_key)
        pred  = pred[t_idx]
        y     = y[t_idx] if isinstance(y, (list, tuple)) else y
    elif isinstance(pred, dict):
        pred = pred[target_key]
        y    = y[target_key] if isinstance(y, dict) else y
    # else: single target tensor, use as-is

    pred = pred.cpu().numpy()  # (n, horizons, quantiles) or (n, quantiles)
    y    = y.cpu().numpy()     # (n, horizons) or (n,)

    if pred.ndim == 3:
        pred_med = pred[:, horizon_idx, 3]   # median quantile index 3
        y_vals   = y[:, horizon_idx]
    else:
        pred_med = pred[:, 3]
        y_vals   = y

    return pred_med, y_vals

def _metrics(pred, actual):
    mae  = float(np.mean(np.abs(pred - actual)))
    rmse = float(np.sqrt(np.mean((pred - actual) ** 2)))
    return mae, rmse

targets_list = cfg['targets']
horizon_indices = EVAL_HORIZON_INDICES if cfg['max_prediction_length'] > 1 else [0]
horizon_labels  = EVAL_HORIZON_LABELS  if cfg['max_prediction_length'] > 1 else [1]

slices = {
    'val':          (val_raw,  slice(None)),
    'test_full':    (test_raw, slice(None)),
    'test_prewar':  (test_raw, prewar_mask),
    'test_war':     (test_raw, war_mask),
}

rows = []
for t in targets_list:
    for h_idx, h_label in zip(horizon_indices, horizon_labels):
        for slice_name, (raw_out, mask) in slices.items():
            pred, actual = _get_pred_actual(raw_out, t, h_idx)
            if not isinstance(mask, slice):
                pred   = pred[mask]
                actual = actual[mask]
            if len(pred) == 0:
                continue
            mae, rmse = _metrics(pred, actual)
            rows.append({'target': t, 'horizon_h': h_label, 'slice': slice_name,
                         'n': len(pred), 'mae': round(mae, 5), 'rmse': round(rmse, 5)})

df_metrics = pd.DataFrame(rows)
df_metrics.to_csv(VARIANT_OUT_DIR / 'metrics.csv', index=False)

print(f'Metrics saved → {VARIANT_OUT_DIR / "metrics.csv"}')
print(df_metrics.to_string(index=False))

## Cell 14 — Variable selection network (VSN) feature importance

In [ ]:
# Try interpret_output first (cleaner API); fall back to direct tensor extraction
try:
    interp = tft_eval.interpret_output(val_raw.output, reduction='mean')
    enc_imp = interp['encoder_variables']
    if isinstance(enc_imp, pd.Series):
        importance_dict = enc_imp.to_dict()
    elif isinstance(enc_imp, dict):
        importance_dict = {k: float(v) for k, v in enc_imp.items()}
    else:
        raise ValueError('Unexpected type from interpret_output')
except Exception as e:
    print(f'interpret_output failed ({e}), using direct tensor extraction')
    enc_vars = val_raw.output['encoder_variables'].cpu().numpy()
    # shape: (n_samples, time_steps, 1, n_features) or (n_samples, 1, n_features)
    if enc_vars.ndim == 4:
        mean_imp = enc_vars.squeeze(2).mean(axis=(0, 1))
    else:
        mean_imp = enc_vars.mean(axis=0)

    # Build feature name list (unknown reals first, then known reals incl. auto-added)
    feat_names = (
        list(train_dataset.time_varying_unknown_reals)
        + list(train_dataset.time_varying_known_reals)
    )
    # TFT adds one extra internal scale feature per target
    n_extra = len(mean_imp) - len(feat_names)
    for i in range(n_extra):
        feat_names.append(f'_internal_{i}')
    importance_dict = {
        name: float(val)
        for name, val in zip(feat_names[:len(mean_imp)], mean_imp)
        if not name.startswith('_internal')
    }

# Sort and save
importance_dict = dict(sorted(importance_dict.items(), key=lambda x: -x[1]))

imp_path = VARIANT_OUT_DIR / 'feature_importance.json'
with open(imp_path, 'w') as f:
    json.dump(importance_dict, f, indent=2)

print(f'Feature importance saved → {imp_path}')
print('Top 10:')
for k, v in list(importance_dict.items())[:10]:
    print(f'  {k:<35} {v:.4f}')

## Cell 15 — Attention weights by lag and sentiment direction

In [ ]:
# encoder_attention shape: (n_samples, n_heads, 1, encoder_length)
enc_attn = val_raw.output['encoder_attention'].cpu().numpy()

# Mean attention over heads and time dimension → (n_samples, encoder_length)
attn_per_sample = enc_attn.mean(axis=(1, 2))   # (n_samples, 48)

# Overall mean attention by lag
mean_attn = attn_per_sample.mean(axis=0)       # (48,) — index 0 = most recent lag

# Sentiment direction split using val sentiment_score
# Each prediction sample's sentiment is from the prediction hour
if hasattr(val_raw, 'index') and val_raw.index is not None:
    _vidx = val_raw.index
    if isinstance(_vidx, pd.DataFrame):
        val_time_idx_arr = _vidx['time_idx'].values
    elif isinstance(_vidx, dict):
        val_time_idx_arr = np.asarray(_vidx['time_idx'])
    else:
        val_time_idx_arr = None
else:
    val_time_idx_arr = None

if val_time_idx_arr is not None:
    val_sent = df.loc[val_time_idx_arr, 'sentiment_score'].values
    val_news  = df.loc[val_time_idx_arr, 'n_articles'].values
else:
    # Approximate: align by sample position
    n_val = len(attn_per_sample)
    val_sent = df['sentiment_score'].values[VAL_START: VAL_START + n_val]
    val_news  = df['n_articles'].values[VAL_START: VAL_START + n_val]

has_news     = val_news > 0
bearish_mask_v = (val_sent < -0.1) & has_news
bullish_mask_v = (val_sent >  0.1) & has_news
neutral_mask_v = (np.abs(val_sent) <= 0.1) & has_news

mean_attn_bearish = attn_per_sample[bearish_mask_v].mean(axis=0) if bearish_mask_v.any() else np.zeros(ENCODER_LENGTH)
mean_attn_bullish = attn_per_sample[bullish_mask_v].mean(axis=0) if bullish_mask_v.any() else np.zeros(ENCODER_LENGTH)
mean_attn_neutral = attn_per_sample[neutral_mask_v].mean(axis=0) if neutral_mask_v.any() else np.zeros(ENCODER_LENGTH)

print(f'Bearish val hours: {bearish_mask_v.sum()}')
print(f'Bullish val hours: {bullish_mask_v.sum()}')
print(f'Neutral val hours: {neutral_mask_v.sum()}')

peak_lag = int(np.argmax(mean_attn[::-1])) - ENCODER_LENGTH
print(f'Overall peak attention lag: {peak_lag}h')

attn_path = VARIANT_OUT_DIR / 'attention.npz'
np.savez(
    attn_path,
    mean_attention         = mean_attn,
    mean_attention_bearish = mean_attn_bearish,
    mean_attention_bullish = mean_attn_bullish,
    mean_attention_neutral = mean_attn_neutral,
)
print(f'Attention arrays saved → {attn_path}')

## Cell 16 — Save all artifacts

In [ ]:
# Predictions dict (val + test)
pred_path = VARIANT_OUT_DIR / 'predictions.pkl'
with open(pred_path, 'wb') as f:
    pickle.dump({'val': val_raw, 'test': test_raw}, f)
print(f'Predictions saved → {pred_path}')

# Run metadata
import subprocess as _sp
try:
    _git_rev = _sp.check_output(['git', 'rev-parse', 'HEAD'],
                                cwd=str(PROJECT_ROOT)).decode().strip()
except Exception:
    _git_rev = 'unavailable'

meta = {
    'ablation_variant':      ABLATION_VARIANT,
    'best_checkpoint':       best_path,
    'best_val_loss':         best_val_loss,
    'best_epoch':            best_epoch,
    'targets':               cfg['targets'],
    'max_prediction_length': cfg['max_prediction_length'],
    'encoder_length':        ENCODER_LENGTH,
    'train_end':             TRAIN_END,
    'val_start':             VAL_START,
    'val_end':               VAL_END,
    'test_start':            TEST_START,
    'test_end':              TEST_END,
    'war_onset_idx':         WAR_ONSET_IDX,
    'war_onset_datetime':    WAR_ONSET_DATETIME,
    'hidden_size':           32,
    'attention_head_size':   4,
    'dropout':               0.1,
    'seed':                  42,
    'total_hours':           TOTAL_HOURS,
    'git_rev':               _git_rev,
}
meta_path = VARIANT_OUT_DIR / 'run_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Metadata saved   → {meta_path}')

## Cell 17 — Summary

In [ ]:
top5 = list(importance_dict.items())[:5]

print('=' * 60)
print(f'  {ABLATION_VARIANT} RUN COMPLETE')
print('=' * 60)
print(f'  Best epoch       : {best_epoch}')
print(f'  Best val_loss    : {best_val_loss:.4f}')

# Test loss (full)
_test_full = df_metrics[(df_metrics['slice'] == 'test_full') &
                        (df_metrics['target'] == cfg['targets'][0]) &
                        (df_metrics['horizon_h'] == 1)]
if len(_test_full):
    print(f'  Test MAE (1h)    : {_test_full["mae"].values[0]:.5f}')

_test_pw = df_metrics[(df_metrics['slice'] == 'test_prewar') &
                      (df_metrics['target'] == cfg['targets'][0]) &
                      (df_metrics['horizon_h'] == 1)]
if len(_test_pw):
    print(f'  Test MAE pre-war : {_test_pw["mae"].values[0]:.5f}')

_test_w = df_metrics[(df_metrics['slice'] == 'test_war') &
                     (df_metrics['target'] == cfg['targets'][0]) &
                     (df_metrics['horizon_h'] == 1)]
if len(_test_w):
    print(f'  Test MAE war     : {_test_w["mae"].values[0]:.5f}')

print(f'  Peak attention lag: {peak_lag}h')
print(f'  Top 5 features:')
for name, val in top5:
    print(f'    {name:<35} {val:.4f}')
print('=' * 60)